In [1]:
!pip uninstall -y langchain langchain-community langchain-core langchain-text-splitters

!pip install -q \
langchain \
langchain-community \
langchain-core \
langchain-text-splitters \
langchain-huggingface \
sentence-transformers \
faiss-cpu \
transformers

Found existing installation: langchain 1.3.4
Uninstalling langchain-1.3.4:
  Successfully uninstalled langchain-1.3.4
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langchain-core 1.4.2
Uninstalling langchain-core-1.4.2:
  Successfully uninstalled langchain-core-1.4.2
Found existing installation: langchain-text-splitters 1.1.2
Uninstalling langchain-text-splitters-1.1.2:
  Successfully uninstalled langchain-text-splitters-1.1.2


In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

/tmp/ipykernel_8935/2807830104.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
knowledge_base = """
Pakistan Climate Overview

Northern Areas:
Northern Pakistan has cold winters and mild summers. Heavy snowfall occurs in winter.

Punjab:
Punjab experiences very hot summers and cool winters. Temperatures often exceed 40°C.

Sindh:
Sindh has a hot desert climate. Karachi has humid coastal weather influenced by the Arabian Sea.

Balochistan:
Balochistan is dry and arid with low rainfall. Quetta experiences cold winters.

Monsoon:
Pakistan receives monsoon rains from July to September, important for agriculture.

Climate Change:
Pakistan is highly vulnerable to floods, heatwaves, and glacier melting.

Floods:
Severe floods occurred in 2010 and 2022 affecting millions of people.
"""

In [4]:
documents = [Document(page_content=knowledge_base)]

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [8]:
def generate_answer(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [9]:
chat_history = []

In [10]:
def climate_chatbot(question):

    global chat_history

    # Retrieve relevant docs
    docs = retriever.invoke(question)

    context = "\n".join([d.page_content for d in docs])

    history = "\n".join(chat_history[-6:])

    prompt = f"""
You are a helpful climate assistant.

Use ONLY the context to answer.

Conversation History:
{history}

Context:
{context}

Question:
{question}

Answer in 1-2 sentences:
"""

    answer = generate_answer(prompt)

    chat_history.append(f"User: {question}")
    chat_history.append(f"Bot: {answer}")

    return answer

In [11]:
print(climate_chatbot("What is the climate of Sindh?"))
print(climate_chatbot("Tell me about Karachi"))
print(climate_chatbot("What are floods in Pakistan?"))
print(climate_chatbot("Explain monsoon season"))

hot desert climate
Karachi has humid coastal weather influenced by the Arabian Sea.
Severe floods occurred in 2010 and 2022 affecting millions of people
Monsoon: Pakistan receives monsoon rains from July to September, important for agriculture.


In [13]:
print("🌦 Pakistan Climate Chatbot Ready!")
print("Type 'exit' to stop\n")

while True:

    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Chat ended.")
        break

    response = climate_chatbot(user_input)

    print("Bot:", response)
    print("-" * 50)

🌦 Pakistan Climate Chatbot Ready!
Type 'exit' to stop

You: What is the climate of Sindh?
Bot: hot desert climate
--------------------------------------------------
You: What are floods in Pakistan?
Bot: Severe floods occurred in 2010 and 2022 affecting millions of people
--------------------------------------------------
You: What is the climate of Punjab?
Bot: very hot summers and cool winters
--------------------------------------------------
You: exit
Chat ended.
